# P1b divergence census on DSBench (ground-truth tasks)\n34 tasks x 5 runs (26 data-analysis questions, 8 data-modelling tasks), no memory, no hint, temperature 1.0. Predictions: `quick_runs/predictions/P1b.md`. Analysis: exact match. Modelling: DSBench relative gap, (score - baseline) / (GT - baseline).

In [1]:
import json, sys, statistics
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd, yaml
REPO = Path.cwd().parent.parent
sys.path.insert(0, str(REPO)); sys.path.insert(0, str(REPO / "quick_runs"))
RUNS = Path("/data/anon/reveries_runs")
TASKS = {t["id"]: t for t in yaml.safe_load(open(REPO / "quick_runs/tasks_p1b.yaml"))["tasks"]}
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)

In [2]:
from metrics import decision_agreement, first_divergence, answer_consistency, consistent_failures, score_spread, task_score
from reveries.decisions import DECISION_TYPES
from score import family_of, SF_CHECKS
DTYPES = [t for t in DECISION_TYPES if t != "final_selection"] + ["final_selection"]
def load_records(probe):
    recs = [json.loads(l) for l in open(RUNS / probe / "records.jsonl")]
    for r in recs:
        r["answer"] = tuple(r["answer"]) if isinstance(r["answer"], list) else r["answer"]
        r["risk"] = TASKS[r["task_id"]]["contamination_risk"]
        r["source"] = "SF" if r["task_id"].startswith("sf_") else "BLADE"
    return recs
def by_task(recs):
    d = defaultdict(list)
    for r in recs: d[r["task_id"]].append(r)
    return dict(sorted(d.items()))
def task_table(groups):
    rows = []
    for t, rs in groups.items():
        row = {"task": t, "risk": TASKS[t]["contamination_risk"], "n": len(rs),
               "answer_consistency": answer_consistency(rs), "score_mean": task_score(rs), "score_sd": score_spread(rs),
               "first_divergence": first_divergence(rs, DTYPES),
               "consistent_failures": ",".join(consistent_failures(rs))}
        for dt in DTYPES:
            row[dt] = decision_agreement(rs, dt)
        rows.append(row)
    return pd.DataFrame(rows)

In [3]:
def load_records(probe):
    recs = [json.loads(l) for l in open(RUNS / probe / "records.jsonl")]
    for r in recs:
        t = TASKS[r["task_id"]]
        r["source"] = "DSA" if t["source"] == "dsbench_analysis" else "DSM"
        r["answer_type"] = t.get("answer_type")
        r["risk"] = t["contamination_risk"]
    return recs

In [4]:
recs = load_records("P1b"); groups = by_task(recs)
print(len(recs), "runs;", Counter(r["stop"] for r in recs))
print({t: [r["run"] for r in rs if r["stop"] != "final_answer"] for t, rs in groups.items() if any(r["stop"] != "final_answer" for r in rs)})
tt = task_table(groups); tt["src"] = tt.task.map(lambda t: "DSA" if t.startswith("dsa_") else "DSM")
tt[["task", "src", "n", "answer_consistency", "score_mean", "score_sd", "first_divergence", "consistent_failures"]]

170 runs; Counter({'final_answer': 166, 'max_steps': 3, 'error: HTTPStatusError("Client error \'400 Bad Request\' for url \'http://localhost:8101/v1/chat/completions\'\\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400")': 1})
{'dsa_00000020_question35': ['seed3'], 'dsa_00000029_question1': ['seed0', 'seed1', 'seed2']}


,task,src,n,answer_consistency,score_mean,score_sd,first_divergence,consistent_failures
0,dsa_00000001_question17,DSA,5,0.6,0.800000,0.447214,framing.target,
1,dsa_00000001_question6,DSA,5,1.0,1.000000,0.000000,framing.population,
2,dsa_00000005_question20,DSA,5,0.6,0.800000,0.447214,framing.population,
3,dsa_00000005_question28,DSA,5,1.0,1.000000,0.000000,framing.population,
4,dsa_00000008_question26,DSA,5,1.0,1.000000,0.000000,variables,
5,dsa_00000010_question1,DSA,5,1.0,1.000000,0.000000,framing.unit,
6,dsa_00000010_question4,DSA,5,1.0,1.000000,0.000000,variables,
7,dsa_00000011_question1,DSA,5,0.6,0.800000,0.447214,framing.population,
8,dsa_00000016_question1,DSA,5,1.0,1.000000,0.000000,framing.target,
9,dsa_00000017_question29,DSA,5,0.4,0.600000,0.547723,framing.unit,


In [5]:
# Analysis questions: per-question answers, accuracy, consistency, consistent wrong answers
dsa = []
for t, rs in groups.items():
    if not t.startswith("dsa_"): continue
    ans = [r["answer"] for r in rs]; c = Counter(a for a in ans if a is not None)
    top, top_n = c.most_common(1)[0] if c else (None, 0)
    dsa.append({"task": t, "type": TASKS[t]["answer_type"], "gold": TASKS[t]["gold"], "answers": ans,
                "n_correct": sum(r["checks"]["correct"] is True for r in rs), "no_answer": sum(a is None for a in ans),
                "answer_consistency": answer_consistency(rs), "modal": top, "modal_n": top_n,
                "all_wrong_same": top_n == len(rs) and all(r["checks"]["correct"] is False for r in rs),
                "all_wrong": all(r["checks"]["correct"] is not True for r in rs)})
dsa = pd.DataFrame(dsa); dsa

,task,type,gold,answers,n_correct,no_answer,answer_consistency,modal,modal_n,all_wrong_same,all_wrong
0,dsa_00000001_question17,integer,1661626,"[472042.0, 1661626.0, 1661626.0, 1661626.0, 1661626.0]",4,0,0.6,1661626.0,4,False,False
1,dsa_00000001_question6,letter,D,"[D, D, D, D, D]",5,0,1.0,D,5,False,False
2,dsa_00000005_question20,letter,B,"[D, B, B, B, B]",4,0,0.6,B,4,False,False
3,dsa_00000005_question28,integer,178052,"[178052.0, 178052.0, 178052.0, 178052.0, 178052.0]",5,0,1.0,178052.0,5,False,False
4,dsa_00000008_question26,letter,C,"[C, C, C, C, C]",5,0,1.0,C,5,False,False
5,dsa_00000010_question1,letter,C,"[C, C, C, C, C]",5,0,1.0,C,5,False,False
6,dsa_00000010_question4,integer,78,"[78.0, 78.0, 78.0, 78.0, 78.0]",5,0,1.0,78.0,5,False,False
7,dsa_00000011_question1,letter,B,"[B, D, B, B, B]",4,0,0.6,B,4,False,False
8,dsa_00000016_question1,letter,D,"[D, D, D, D, D]",5,0,1.0,D,5,False,False
9,dsa_00000017_question29,letter,E,"[C, E, E, E, C]",3,0,0.4,E,3,False,False


In [6]:
acc = dsa.assign(acc=dsa.n_correct / 5).groupby("type").agg(questions=("task", "size"), accuracy=("acc", "mean"),
    answer_consistency=("answer_consistency", "mean"), all5_correct=("n_correct", lambda x: int((x == 5).sum())),
    all5_wrong=("all_wrong", "sum"), all5_same_wrong=("all_wrong_same", "sum"))
pooled = {"accuracy": round(dsa.n_correct.sum() / (5 * len(dsa)), 3), "answer_consistency": round(dsa.answer_consistency.mean(), 3)}
print(pooled); acc.round(3)

{'accuracy': np.float64(0.862), 'answer_consistency': np.float64(0.812)}


,questions,accuracy,answer_consistency,all5_correct,all5_wrong,all5_same_wrong
type,,,,,,
integer,6,0.767,0.700,3,1,0
letter,20,0.890,0.845,15,0,0


In [7]:
# DSBench label error: competition 00000020 lists 10 answers for questions 28-37, but question 35 (a letter
# question, options A-I) has the number 14978914492, which belongs to a later question. Its gold is unusable.
LABEL_ERRORS = ["dsa_00000020_question35"]
clean = dsa[~dsa.task.isin(LABEL_ERRORS)]
acc_clean = clean.assign(acc=clean.n_correct / 5).groupby("type").agg(questions=("task", "size"), accuracy=("acc", "mean"),
    answer_consistency=("answer_consistency", "mean"), all5_correct=("n_correct", lambda x: int((x == 5).sum())), all5_wrong=("all_wrong", "sum"))
pooled_clean = {"questions": len(clean), "accuracy": round(clean.n_correct.sum() / (5 * len(clean)), 3),
                "answer_consistency": round(clean.answer_consistency.mean(), 3)}
print(pooled_clean); acc_clean.round(3)

{'questions': 25, 'accuracy': np.float64(0.896), 'answer_consistency': np.float64(0.844)}


,questions,accuracy,answer_consistency,all5_correct,all5_wrong
type,,,,,
integer,5,0.92,0.840,3,0
letter,20,0.89,0.845,15,0


In [8]:
# Is consistency tied to correctness? (questions where all 5 runs agree)
agree = dsa[dsa.answer_consistency == 1.0]
print(len(agree), "questions with all 5 answers identical;", int((agree.n_correct == 5).sum()), "all right,",
      int((agree.n_correct == 0).sum()), "all wrong")

18 questions with all 5 answers identical; 18 all right, 0 all wrong


In [9]:
# Modelling tasks: relative gap per run, spread, valid submissions
dsm = []
for r in recs:
    if r["source"] != "DSM": continue
    dsm.append({"task": r["task_id"], "run": r["run"], "stop": r["stop"], "valid": r["checks"]["valid_submission"],
                "raw_metric": r.get("raw_metric"), "rel_gap": r["score"], "score_error": (r.get("score_error") or "")[:80],
                "family": r["executed"].get("model.family"), "all_families": r["executed"].get("model.all_families"),
                "validation": r["executed"].get("validation"), "steps": r["steps"]})
dsm = pd.DataFrame(dsm)
dsm_task = dsm.groupby("task").agg(valid=("valid", "sum"), rel_gap_mean=("rel_gap", "mean"), rel_gap_sd=("rel_gap", "std"),
    rel_gap_min=("rel_gap", "min"), rel_gap_max=("rel_gap", "max"), families=("family", lambda x: dict(Counter(x))),
    validation=("validation", lambda x: dict(Counter(x))))
print("valid submissions:", int(dsm.valid.sum()), "of", len(dsm)); dsm_task.round(3)

valid submissions: 40 of 40


,valid,rel_gap_mean,rel_gap_sd,rel_gap_min,rel_gap_max,families,validation
task,,,,,,,
dsm_bike-sharing-demand,5,0.938,0.005,0.932,0.942,{'xgboost': 5},{'holdout_split': 5}
dsm_playground-series-s3e12,5,0.719,0.021,0.688,0.743,"{'random_forest': 4, 'other': 1}","{'stratified_kfold_cv': 4, 'grid_search_cv': 1}"
dsm_playground-series-s3e17,5,0.919,0.005,0.914,0.927,{'xgboost': 5},"{'stratified_kfold_cv': 3, 'holdout_split': 2}"
dsm_playground-series-s3e23,5,0.578,0.001,0.577,0.579,"{'xgboost': 4, 'other': 1}",{'stratified_kfold_cv': 5}
dsm_playground-series-s3e5,5,0.493,0.009,0.482,0.501,"{'random_forest': 1, 'other': 1, 'xgboost': 3}","{'stratified_kfold_cv': 1, 'kfold_cv': 4}"
dsm_playground-series-s3e8,5,0.857,0.000,0.857,0.857,{'xgboost': 5},"{'kfold_cv': 4, 'holdout_split': 1}"
dsm_playground-series-s4e3,5,0.768,0.007,0.759,0.776,"{'xgboost': 4, 'gradient_boosting': 1}","{'holdout_split': 2, 'stratified_kfold_cv': 2, 'kfold_cv': 1}"
dsm_tabular-playground-series-apr-2021,5,0.603,0.002,0.601,0.606,"{'xgboost': 4, 'random_forest': 1}","{'stratified_kfold_cv': 3, 'kfold_cv': 2}"


In [10]:
# Decision agreement by type, split by part (mean over tasks)
rank = pd.DataFrame({"all": tt[DTYPES].mean(), "DSA": tt[tt.src == "DSA"][DTYPES].mean(), "DSM": tt[tt.src == "DSM"][DTYPES].mean(),
                     "tasks_scored_DSA": tt[tt.src == "DSA"][DTYPES].notna().sum(), "tasks_scored_DSM": tt[tt.src == "DSM"][DTYPES].notna().sum()}).sort_values("all")
print(Counter(tt[tt.src == "DSA"].first_divergence), Counter(tt[tt.src == "DSM"].first_divergence)); rank.round(2)

Counter({'framing.target': 8, 'framing.population': 6, 'variables': 5, 'framing.unit': 5, 'metric': 1, 'transformation': 1}) Counter({'framing.unit': 4, 'variables': 2, 'framing.target': 1, 'model.family': 1})


,all,DSA,DSM,tasks_scored_DSA,tasks_scored_DSM
transformation,0.210667,0.22037,0.185714,18,7
final_selection,0.253922,0.251282,0.2625,26,8
variables,0.288889,0.189583,0.4875,16,8
framing.unit,0.542157,0.512821,0.6375,26,8
framing.target,0.6,0.428571,0.942857,14,7
framing.population,0.794118,0.730769,1.0,26,8
metric,0.822222,0.784848,0.925,22,8
verdict,0.857143,0.857143,NaN,7,0
validation,0.905882,1.0,0.6,26,8
model.family,0.923333,1.0,0.7125,22,8


In [11]:
# Analysis questions: on questions where runs disagree, which decision types differ?
dis = tt[(tt.src == "DSA") & (tt.answer_consistency < 1.0)]
pd.DataFrame({"disagreeing_q": dis[DTYPES].mean(), "agreeing_q": tt[(tt.src == "DSA") & (tt.answer_consistency == 1.0)][DTYPES].mean()}).round(2)

,disagreeing_q,agreeing_q
framing.target,0.122222,0.512121
framing.population,0.4875,0.838889
framing.unit,0.491667,0.522222
variables,0.0,0.252778
transformation,0.183333,0.238889
model.family,1.0,1.0
model.controls,NaN,NaN
validation,1.0,1.0
metric,0.866667,0.760784
verdict,1.0,0.833333


In [12]:
# bike-sharing-demand: do runs use casual / registered (in train only; they sum to the target)?
def code_cells(r):
    return [json.loads(l) for l in open(RUNS / "P1b" / r["task_id"] / r["run"] / "events.jsonl") if json.loads(l)["event"] == "code_executed"]
bike = []
for r in groups["dsm_bike-sharing-demand"]:
    cells = code_cells(r)
    feat = [c for c in cells if ("casual" in c["code"] or "registered" in c["code"])]
    bike.append({"run": r["run"], "cells_mentioning": len(feat),
                 "dropped": any(("drop" in c["code"] and ("casual" in c["code"] or "registered" in c["code"])) for c in cells),
                 "rel_gap": r["score"], "extractor_variables_has_them": bool({"casual", "registered"} & set(r["executed"].get("variables") or []))})
pd.DataFrame(bike)

,run,cells_mentioning,dropped,rel_gap,extractor_variables_has_them
0,seed0,4,True,0.933137,False
1,seed1,2,True,0.940676,False
2,seed2,5,True,0.942359,True
3,seed3,0,False,0.940909,False
4,seed4,0,False,0.932182,False


In [13]:
# Behaviour: how the agent works
def run_events(r):
    return [json.loads(l) for l in open(RUNS / "P1b" / r["task_id"] / r["run"] / "events.jsonl")]
beh = []
for r in recs:
    ev = run_events(r); cells = [e for e in ev if e["event"] == "code_executed"]
    beh.append({"task": r["task_id"], "source": r["source"], "run": r["run"], "stop": r["stop"], "steps": r["steps"],
        "code_cells": len(cells), "cell_errors": sum(not c["ok"] for c in cells),
        "stated_types": len(set(d["type"] for d in r["stated"])), "silent_types": len(r["executed"].get("silent") or []),
        "completion_tokens": r["tokens"]["completion"], "minutes": (ev[-1]["t"] - ev[0]["t"]) / 60})
beh = pd.DataFrame(beh)
beh.groupby("source")[["steps", "code_cells", "cell_errors", "stated_types", "silent_types", "completion_tokens", "minutes"]].describe().T.round(1)

source                       DSA      DSM
steps             count    130.0     40.0
                  mean       6.2      6.9
                  std        5.2      2.1
                  min        2.0      3.0
                  25%        2.0      5.0
                  50%        4.0      7.0
                  75%        9.0      8.2
                  max       25.0     11.0
code_cells        count    130.0     40.0
                  mean       4.2      5.7
                  std        4.3      1.9
                  min        1.0      2.0
                  25%        1.0      4.0
                  50%        2.0      6.0
                  75%        6.0      7.0
                  max       21.0      9.0
cell_errors       count    130.0     40.0
                  mean       0.2      0.8
                  std        0.6      0.7
                  min        0.0      0.0
                  25%        0.0      0.0
                  50%        0.0      1.0
                  75%        0.0      1.0
                  max        4.0      2.0
stated_types      count    130.0     40.0
                  mean       3.6      8.5
                  std        2.1      1.3
                  min        0.0      4.0
                  25%        2.0      8.0
                  50%        3.0      8.5
                  75%        5.0     10.0
                  max       10.0     10.0
silent_types      count    130.0     40.0
                  mean       8.0      1.0
                  std        2.6      1.2
                  min        2.0      0.0
                  25%        6.0      0.0
                  50%        8.0      1.0
                  75%       10.0      1.2
                  max       13.0      7.0
completion_tokens count    130.0     40.0
                  mean    9669.5   6113.3
                  std     8866.9   3665.1
                  min      442.0   2608.0
                  25%     2306.5   3551.0
                  50%     7541.5   4662.0
                  75%    13984.2   7393.5
                  max    39616.0  16145.0
minutes           count    130.0     40.0
                  mean      41.2     37.5
                  std       37.0     15.7
                  min        1.6     11.9
                  25%       13.8     25.9
                  50%       28.9     35.7
                  75%       62.4     46.6
                  max      154.4     71.4

In [14]:
summary = {"runs": len(recs), "stops": dict(Counter(r["stop"] for r in recs)),
  "dsa_pooled": pooled, "dsa_by_type": acc.round(3).to_dict("index"), "dsa_label_errors": LABEL_ERRORS,
  "dsa_pooled_clean": pooled_clean, "dsa_by_type_clean": acc_clean.round(3).to_dict("index"),
  "dsa_all_agree": [len(agree), int((agree.n_correct == 5).sum()), int((agree.n_correct == 0).sum())],
  "dsa_consistent_wrong_same": sorted(dsa[dsa.all_wrong_same].task), "dsa_all_wrong": sorted(dsa[dsa.all_wrong].task),
  "dsa_table": dsa.to_dict("records"),
  "dsm_valid": [int(dsm.valid.sum()), len(dsm)], "dsm_tasks_sd_above_0.02": int((dsm_task.rel_gap_sd > 0.02).sum()),
  "dsm_per_task": dsm_task.round(4).to_dict("index"),
  "agreement_rank": rank[["all", "DSA", "DSM"]].round(3).to_dict(),
  "first_divergence_DSA": dict(Counter(tt[tt.src == "DSA"].first_divergence)), "first_divergence_DSM": dict(Counter(tt[tt.src == "DSM"].first_divergence)),
  "bike_casual_registered": bike,
  "behaviour_by_source": beh.groupby("source")[["steps", "code_cells", "cell_errors", "silent_types", "minutes"]].mean().round(2).to_dict("index"),
  "per_task": tt.set_index("task").to_dict("index")}
(REPO / "quick_runs/results/P1b").mkdir(parents=True, exist_ok=True)
json.dump(summary, open(REPO / "quick_runs/results/P1b/summary.json", "w"), indent=1, default=str)
dsa.to_csv(REPO / "quick_runs/results/P1b/analysis.csv", index=False); dsm.to_csv(REPO / "quick_runs/results/P1b/modeling.csv", index=False)
beh.to_csv(REPO / "quick_runs/results/P1b/behaviour.csv", index=False)
"done" 

'done'